# PINN v4 — Corrected Preprocessing + E(x) + Checkpointing

## Changes from v3:
1. **Preprocessing fix**: Smooth Re(H) and Im(H) separately (not magnitude & phase)
2. **E(x) bounding**: Sigmoid reparameterization keeps E(x) in [50, 400] GPa
3. **Smoother LR scheduling**: CosineAnnealingWarmRestarts replaces ReduceLROnPlateau (eliminates spikes)
4. **Improved parameter learning rates**: Higher LR for scalar params to prevent stagnation
5. **Google Drive checkpointing**: Adam + L-BFGS phases both save to Drive
6. **Beam params**: L=126.06mm, h=0.95mm, b=10.13mm, rho=8216 kg/m³

## 1. Imports and device setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from google.colab import drive
import os, json, glob, shutil

# ── Float64 everywhere — non-negotiable for 4th-order autograd ───────────
DTYPE = torch.float64
torch.set_default_dtype(DTYPE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Precision: {DTYPE}")

## 2. Mount Google Drive and set checkpoint paths

In [ ]:
drive.mount('/content/drive')

CKPT_DIR = "/content/drive/MyDrive/PINN_v4_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Checkpoint directory: {CKPT_DIR}")

ADAM_CKPT_EVERY  = 2000   # save Adam checkpoint every N epochs
LBFGS_CKPT_EVERY = 30     # save L-BFGS checkpoint every N steps

## 3. Physical constants

Beam geometry and material properties. These are **fixed and known** — not identified.
- x=0 is the clamped end (with spring compliance)
- x=L is the free end (with tip mass and applied force)

In [ ]:
# ── Beam geometry ─────────────────────────────────────────────────────────
L   = 126.06e-3    # beam length [m]
b   = 10.13e-3     # beam width [m]
h   = 0.95e-3      # beam thickness [m]
rho = 8216.0       # density [kg/m³]

# ── Derived constants ────────────────────────────────────────────────────
A = b * h                      # cross-sectional area [m²]
I_beam = b * h**3 / 12         # second moment of area [m⁴]

# ── Tip mass (set to 0 if no object attached) ────────────────────────────
m_tip = 0.0    # [kg] — change this per experiment

# ── Measurement location ─────────────────────────────────────────────────
x_meas = L     # measurement at the free tip (x = L)

# ── Frequency range ──────────────────────────────────────────────────────
MIN_FREQ_HZ = 1.0
MAX_FREQ_HZ = 5000.0

print(f"Beam: L={L*1e3:.2f} mm, b={b*1e3:.2f} mm, h={h*1e3:.2f} mm")
print(f"A={A:.4e} m², I={I_beam:.4e} m⁴, rho={rho:.1f} kg/m³")
print(f"Frequency range: {MIN_FREQ_HZ}–{MAX_FREQ_HZ} Hz")

## 4. Load experimental FRF data

Upload your `.npz` or `.mat` file. The loader expects:
- `f` or `freq`: frequency vector [Hz]
- `V` or `frf`: complex FRF (velocity / force)

Adjust the loader if your file format differs.

In [ ]:
from google.colab import files as colab_files

# ── Upload ────────────────────────────────────────────────────────────────
uploaded = colab_files.upload()
uploaded_name = list(uploaded.keys())[0]
print(f"Uploaded: {uploaded_name}")

# ── Generic loader ────────────────────────────────────────────────────────
def load_frf(filename, x_meas=L):
    ext = os.path.splitext(filename)[1].lower()
    if ext == '.npz':
        data = np.load(filename, allow_pickle=True)
        keys = list(data.keys())
        print(f"  Keys in file: {keys}")
        f_key = next((k for k in keys if k.lower() in ['f', 'freq', 'frequency', 'f_vec']), keys[0])
        v_key = next((k for k in keys if k.lower() in ['v', 'frf', 'h', 'velocity', 'v_exp']), keys[1])
        f_hz = data[f_key].real.flatten()
        V    = data[v_key].flatten()
    elif ext == '.mat':
        import scipy.io as sio
        data = sio.loadmat(filename)
        keys = [k for k in data.keys() if not k.startswith('__')]
        print(f"  Keys in file: {keys}")
        f_key = next((k for k in keys if 'f' in k.lower()), keys[0])
        v_key = next((k for k in keys if 'v' in k.lower() or 'h' in k.lower() or 'frf' in k.lower()), keys[1])
        f_hz = data[f_key].real.flatten()
        V    = data[v_key].flatten()
    else:
        raise ValueError(f"Unsupported format: {ext}. Use .npz or .mat")

    x_arr = np.full_like(f_hz, x_meas, dtype=float)
    return f_hz, x_arr, V

f_raw, x_raw, V_raw = load_frf(uploaded_name, x_meas=x_meas)
print(f"Loaded {len(f_raw)} points, freq range: {f_raw.min():.1f} – {f_raw.max():.1f} Hz")

## 5. Preprocessing — CORRECTED: Smooth Re and Im separately

### What went wrong in v3:
v3 smoothed **magnitude** and **phase** independently with Savitzky-Golay filters:
```python
V_mag_s   = savgol_filter(|V|, ...)
V_phase_s = savgol_filter(angle(V), ...)
V_smooth  = V_mag_s * exp(j * V_phase_s)
```
This destroyed the phase information because:
- Phase has ±π wrapping discontinuities at every resonance
- SG filter bridges across these discontinuities, flattening phase toward zero
- The imaginary part of the reconstructed V_smooth is then ~0 everywhere
- The PINN data loss effectively becomes magnitude-only

### The fix in v4:
Smooth the **real** and **imaginary** parts separately:
```python
V_re_s = savgol_filter(Re(V), ...)
V_im_s = savgol_filter(Im(V), ...)
V_smooth = V_re_s + j * V_im_s
```
Re(V) and Im(V) are smooth, continuous functions — no wrapping, no discontinuities.
This preserves both magnitude AND phase information naturally.

In [ ]:
eps_db = 1e-30  # for safe dB conversion

# ── Hard frequency cutoff ─────────────────────────────────────────────────
mask = (f_raw >= MIN_FREQ_HZ) & (f_raw <= MAX_FREQ_HZ)
f_cut = f_raw[mask]
V_cut = V_raw[mask]
print(f"After cutoff: {len(f_cut)} points  ({f_cut.min():.1f} – {f_cut.max():.1f} Hz)")

# ── Savitzky-Golay smoothing on REAL and IMAGINARY parts separately ───────
sg_win = min(51, (len(f_cut) // 20) * 2 + 1)
sg_win = max(sg_win, 5)

V_re_smooth = savgol_filter(V_cut.real, window_length=sg_win, polyorder=3)
V_im_smooth = savgol_filter(V_cut.imag, window_length=sg_win, polyorder=3)
V_smooth    = V_re_smooth + 1j * V_im_smooth

print(f"SG window: {sg_win} points")

# ── Subsample to ~300 points ─────────────────────────────────────────────
N_train = 300
idx     = np.linspace(0, len(f_cut) - 1, N_train, dtype=int)
f_train = f_cut[idx]
V_train = V_smooth[idx]
x_train = np.full_like(f_train, x_meas)
print(f"Training points: {len(f_train)}")

# ── Diagnostic plot: verify phase is preserved ───────────────────────────
fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axs[0].plot(f_cut,   20*np.log10(np.abs(V_cut) + eps_db),   alpha=0.35, label="Raw")
axs[0].plot(f_train, 20*np.log10(np.abs(V_train) + eps_db), "r-", lw=1.8, label="Smoothed + subsampled")
axs[0].set_ylabel("Magnitude [dB]")
axs[0].legend()
axs[0].set_title("Experimental FRF preprocessing (v4: Re/Im smoothing)")

axs[1].plot(f_cut,   np.angle(V_cut),   alpha=0.35, label="Raw")
axs[1].plot(f_train, np.angle(V_train), "r-", lw=1.8, label="Smoothed + subsampled")
axs[1].set_ylabel("Phase [rad]")
axs[1].set_xlabel("Frequency [Hz]")
axs[1].legend()

plt.tight_layout()
plt.show()

print("\n>> CHECK: The red phase curve should show clear transitions at resonances,")
print(">>        NOT a flat line near zero. If it's flat, something is still wrong.")

## 6. Convert to tensors and normalize

Inputs are normalized to [-1, 1] so the network receives balanced gradient signal
across all frequencies. Without this, high-frequency points dominate training.

In [ ]:
# ── Data tensors ──────────────────────────────────────────────────────────
omega_train = 2 * np.pi * f_train     # angular frequency [rad/s]

# Velocity FRF: V = iω W  →  V_real = -ω W_imag,  V_imag = ω W_real
Vr_np = V_train.real
Vi_np = V_train.imag

omega_data = torch.tensor(omega_train, dtype=DTYPE, device=device).unsqueeze(1)
x_data     = torch.tensor(x_train,     dtype=DTYPE, device=device).unsqueeze(1)
Vr_data    = torch.tensor(Vr_np,       dtype=DTYPE, device=device).unsqueeze(1)
Vi_data    = torch.tensor(Vi_np,       dtype=DTYPE, device=device).unsqueeze(1)

# ── Normalization bounds ─────────────────────────────────────────────────
x_min, x_max         = 0.0, L
omega_min, omega_max = omega_data.min().item(), omega_data.max().item()

def normalize_x(x):
    return 2.0 * (x - x_min) / (x_max - x_min) - 1.0

def normalize_omega(w):
    return 2.0 * (w - omega_min) / (omega_max - omega_min) - 1.0

# ── Collocation grid ─────────────────────────────────────────────────────
N_x, N_omega = 50, 50
x_coll_np     = np.linspace(0, L, N_x)
omega_coll_np = np.linspace(omega_min, omega_max, N_omega)
xx, ww        = np.meshgrid(x_coll_np, omega_coll_np)

x_col     = torch.tensor(xx.flatten(),  dtype=DTYPE, device=device).unsqueeze(1).requires_grad_(True)
omega_col = torch.tensor(ww.flatten(),  dtype=DTYPE, device=device).unsqueeze(1)

print(f"Collocation grid: {N_x}x{N_omega} = {len(x_col)} points")
print(f"omega range: [{omega_min:.1f}, {omega_max:.1f}] rad/s")

## 7. Neural network architecture

**WNet**: Takes (x, omega) normalized to [-1,1], outputs (W_real, W_imag)

**ENet**: Takes x normalized to [-1,1], outputs E(x) bounded in [E_min, E_max] GPa

### E(x) bounding — NEW in v4
In v3, ENet was unbounded and E(x) blew up to 4400 GPa.
Now we use sigmoid reparameterization:
```
E(x) = E_min + (E_max - E_min) * sigmoid(raw_output)
```
This guarantees E(x) stays in a physically plausible range.

In [ ]:
# ── E(x) bounds [Pa] ─────────────────────────────────────────────────────
E_MIN = 50e9    # 50 GPa — softest plausible metal
E_MAX = 400e9   # 400 GPa — hardest plausible metal/ceramic
E_INIT = 200e9  # initial guess: ~steel


class WNet(nn.Module):
    """Displacement field: (x_norm, omega_norm) -> (W_real, W_imag)"""
    def __init__(self, hidden=128, layers=4):
        super().__init__()
        net = [nn.Linear(2, hidden), nn.Tanh()]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.Tanh()]
        net += [nn.Linear(hidden, 2)]
        self.net = nn.Sequential(*net)

    def forward(self, x_norm, omega_norm):
        inp = torch.cat([x_norm, omega_norm], dim=1)
        out = self.net(inp)
        return out[:, 0:1], out[:, 1:2]  # W_real, W_imag


class ENet(nn.Module):
    """Spatially varying Young's modulus: x_norm -> E(x) in [E_MIN, E_MAX]"""
    def __init__(self, hidden=64, layers=3):
        super().__init__()
        net = [nn.Linear(1, hidden), nn.Tanh()]
        for _ in range(layers - 1):
            net += [nn.Linear(hidden, hidden), nn.Tanh()]
        net += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*net)

        # Initialize bias of last layer so sigmoid output ~ E_INIT
        with torch.no_grad():
            target_sigmoid = (E_INIT - E_MIN) / (E_MAX - E_MIN)
            init_bias = np.log(target_sigmoid / (1.0 - target_sigmoid))
            self.net[-1].bias.fill_(init_bias)
            self.net[-1].weight.fill_(0.0)

    def forward(self, x_norm):
        raw = self.net(x_norm)
        return E_MIN + (E_MAX - E_MIN) * torch.sigmoid(raw)


# ── Instantiate ──────────────────────────────────────────────────────────
w_net = WNet(hidden=128, layers=4).to(device)
e_net = ENet(hidden=64,  layers=3).to(device)

n_w = sum(p.numel() for p in w_net.parameters())
n_e = sum(p.numel() for p in e_net.parameters())
print(f"WNet parameters: {n_w:,}")
print(f"ENet parameters: {n_e:,}")

## 8. Trainable physical parameters

These are optimized alongside the network weights:
- **eta**: loss factor (damping)
- **kx**: translational spring stiffness at x=0
- **kphi**: rotational spring stiffness at x=0
- **F**: excitation force magnitude

Each uses a raw (unconstrained) parameter + scale factor to control the mapping.

In [ ]:
# ── Initial guesses and scales ────────────────────────────────────────────
eta_init  = 0.01          # loss factor — typical for metals
kx_init   = 1e5           # translational spring [N/m]
kphi_init = 1e3           # rotational spring [N*m/rad]
F_init    = 1.0           # force amplitude [N]

kx_scale   = 1e3          # raw * scale = kx
kphi_scale = 1e1          # raw * scale = kphi

# ── Raw trainable parameters ─────────────────────────────────────────────
eta_raw  = nn.Parameter(torch.tensor(eta_init,                dtype=DTYPE, device=device))
kx_raw   = nn.Parameter(torch.tensor(kx_init / kx_scale,     dtype=DTYPE, device=device))
kphi_raw = nn.Parameter(torch.tensor(kphi_init / kphi_scale, dtype=DTYPE, device=device))
F_raw    = nn.Parameter(torch.tensor(F_init,                  dtype=DTYPE, device=device))

phys_params = [eta_raw, kx_raw, kphi_raw, F_raw]


def get_params():
    """Return physical values (with positivity enforced where needed)."""
    eta  = torch.abs(eta_raw)
    kx   = torch.abs(kx_raw) * kx_scale
    kphi = torch.abs(kphi_raw) * kphi_scale
    F    = F_raw  # force can be positive or negative
    return eta, kx, kphi, F


eta_v, kx_v, kphi_v, F_v = get_params()
print(f"Initial: eta={eta_v.item():.4f}, kx={kx_v.item():.1f}, kphi={kphi_v.item():.1f}, F={F_v.item():.4f}")

## 9. Autograd derivative helper

In [ ]:
def grad(y, x):
    """Compute dy/dx with graph retained for higher-order derivatives."""
    return torch.autograd.grad(
        y, x,
        grad_outputs=torch.ones_like(y),
        create_graph=True,
        retain_graph=True,
    )[0]

## 10. Physics-informed loss function

### PDE (variable E(x)):
```
E(x)*W'''' + 2*E'(x)*W''' + E''(x)*W'' - (rhoA/I)*omega^2*W = 0
```
With complex modulus E*(x) = E(x)*(1 + j*eta):
- Real residual = I*[E*Wr'''' + 2E'*Wr''' + E''*Wr'' - eta(E*Wi'''' + 2E'*Wi''' + E''*Wi'')] - rhoA*omega^2*Wr
- Imag residual = I*[E*Wi'''' + 2E'*Wi''' + E''*Wi'' + eta(E*Wr'''' + 2E'*Wr''' + E''*Wr'')] - rhoA*omega^2*Wi

### Boundary conditions:
- x=0: EI*W''(0) + kphi*W'(0) = 0  (rotational spring)
- x=0: EI*W'''(0) + kx*W(0) = 0  (translational spring)
- x=L: W''(L) = 0  (zero moment at free end)
- x=L: EI*W'''(L) + m_tip*omega^2*W(L) + F = 0  (tip mass + force)

### Data loss:
V = i*omega*W  ->  V_real = -omega*W_imag,  V_imag = omega*W_real

In [ ]:
A_t     = torch.tensor(A,      dtype=DTYPE, device=device)
I_t     = torch.tensor(I_beam, dtype=DTYPE, device=device)
rho_t   = torch.tensor(rho,    dtype=DTYPE, device=device)
m_tip_t = torch.tensor(m_tip,  dtype=DTYPE, device=device)


def pinn_loss(w_pde=10.0, w_bc=100.0, w_data=1.0):
    eta, kx, kphi, F = get_params()

    # ── Forward pass through WNet at collocation points ──────────────────
    x_n = normalize_x(x_col)
    w_n = normalize_omega(omega_col)
    Wr, Wi = w_net(x_n, w_n)

    # ── E(x) and its spatial derivatives via ENet ────────────────────────
    x_col_E = x_col.detach().requires_grad_(True)
    x_n_E   = normalize_x(x_col_E)
    E_val   = e_net(x_n_E)                # E(x)
    E_x     = grad(E_val, x_col_E)        # dE/dx
    E_xx    = grad(E_x,   x_col_E)        # d2E/dx2

    # ── Spatial derivatives of W ─────────────────────────────────────────
    Wr_x    = grad(Wr, x_col);    Wi_x    = grad(Wi, x_col)
    Wr_xx   = grad(Wr_x, x_col);  Wi_xx   = grad(Wi_x, x_col)
    Wr_xxx  = grad(Wr_xx, x_col);  Wi_xxx  = grad(Wi_xx, x_col)
    Wr_xxxx = grad(Wr_xxx, x_col); Wi_xxxx = grad(Wi_xxx, x_col)

    omega2 = omega_col ** 2

    # ── PDE residual ─────────────────────────────────────────────────────
    # Real beam operator applied to Wr and Wi
    D_Wr = E_val * Wr_xxxx + 2 * E_x * Wr_xxx + E_xx * Wr_xx
    D_Wi = E_val * Wi_xxxx + 2 * E_x * Wi_xxx + E_xx * Wi_xx

    r_pde_r = I_t * (D_Wr - eta * D_Wi) - rho_t * A_t * omega2 * Wr
    r_pde_i = I_t * (D_Wi + eta * D_Wr) - rho_t * A_t * omega2 * Wi

    pde_loss = torch.mean(r_pde_r**2 + r_pde_i**2)

    # ── Boundary conditions ──────────────────────────────────────────────
    omega_b = omega_col.detach()
    w_n_b   = normalize_omega(omega_b)

    # -- x = 0 (clamped with springs) --
    x0   = torch.zeros_like(omega_b, requires_grad=True)
    x0_n = normalize_x(x0)
    Wr0, Wi0 = w_net(x0_n, w_n_b)

    Wr0_x   = grad(Wr0, x0);    Wi0_x   = grad(Wi0, x0)
    Wr0_xx  = grad(Wr0_x, x0);  Wi0_xx  = grad(Wi0_x, x0)
    Wr0_xxx = grad(Wr0_xx, x0); Wi0_xxx = grad(Wi0_xx, x0)

    # E at x=0
    x0_E   = torch.zeros(1, 1, dtype=DTYPE, device=device, requires_grad=False)
    E0_val = e_net(normalize_x(x0_E)).detach()
    EI_r_0 = E0_val * I_t
    EI_i_0 = eta * E0_val * I_t

    # BC1: E*I*W''(0) + kphi*W'(0) = 0
    bc0_m_r = (EI_r_0 * Wr0_xx - EI_i_0 * Wi0_xx) + kphi * Wr0_x
    bc0_m_i = (EI_r_0 * Wi0_xx + EI_i_0 * Wr0_xx) + kphi * Wi0_x
    # BC2: E*I*W'''(0) + kx*W(0) = 0
    bc0_v_r = (EI_r_0 * Wr0_xxx - EI_i_0 * Wi0_xxx) + kx * Wr0
    bc0_v_i = (EI_r_0 * Wi0_xxx + EI_i_0 * Wr0_xxx) + kx * Wi0

    # -- x = L (free end with tip mass + force) --
    xL   = torch.full_like(omega_b, fill_value=L, requires_grad=True)
    xL_n = normalize_x(xL)
    WrL, WiL = w_net(xL_n, w_n_b)

    WrL_x   = grad(WrL, xL);    WiL_x   = grad(WiL, xL)
    WrL_xx  = grad(WrL_x, xL);  WiL_xx  = grad(WiL_x, xL)
    WrL_xxx = grad(WrL_xx, xL); WiL_xxx = grad(WiL_xx, xL)

    # E at x=L
    xL_E   = torch.full((1, 1), L, dtype=DTYPE, device=device, requires_grad=False)
    EL_val = e_net(normalize_x(xL_E)).detach()
    EI_r_L = EL_val * I_t
    EI_i_L = eta * EL_val * I_t

    # BC3: W''(L) = 0  (zero moment at free end)
    bcL_m_r = WrL_xx
    bcL_m_i = WiL_xx
    # BC4: E*I*W'''(L) + m_tip*omega^2*W(L) + F = 0
    bcL_v_r = (EI_r_L * WrL_xxx - EI_i_L * WiL_xxx) + m_tip_t * omega_b**2 * WrL + F
    bcL_v_i = (EI_r_L * WiL_xxx + EI_i_L * WrL_xxx) + m_tip_t * omega_b**2 * WiL

    bc_loss = torch.mean(
        bc0_m_r**2 + bc0_m_i**2 +
        bc0_v_r**2 + bc0_v_i**2 +
        bcL_m_r**2 + bcL_m_i**2 +
        bcL_v_r**2 + bcL_v_i**2
    )

    # ── Data loss: V = i*omega*W -> V_r = -omega*Wi, V_i = omega*Wr ──────
    x_d_n = normalize_x(x_data)
    w_d_n = normalize_omega(omega_data)
    Wr_d, Wi_d = w_net(x_d_n, w_d_n)

    Vp_r = -omega_data * Wi_d
    Vp_i =  omega_data * Wr_d

    data_loss = torch.mean((Vp_r - Vr_data)**2 + (Vp_i - Vi_data)**2)

    total = w_pde * pde_loss + w_bc * bc_loss + w_data * data_loss

    logs = {
        "loss": total.detach().item(),
        "pde":  pde_loss.detach().item(),
        "bc":   bc_loss.detach().item(),
        "data": data_loss.detach().item(),
    }
    return total, logs

## 11. Optimizer setup

### Changes from v3:
- **CosineAnnealingWarmRestarts** instead of ReduceLROnPlateau — eliminates the periodic loss spikes
- **Higher LR for scalar params** (1e-3 vs 5e-4) — kx was stuck because its gradient signal was too small relative to LR
- Three parameter groups: WNet, ENet, scalar physical params

In [ ]:
ADAM_EPOCHS = 20000
ADAM_LR_NET  = 1e-3   # for WNet and ENet
ADAM_LR_PHYS = 1e-3   # for eta, kx, kphi, F — increased from v3's 5e-4

opt_adam = torch.optim.Adam([
    {"params": w_net.parameters(), "lr": ADAM_LR_NET},
    {"params": e_net.parameters(), "lr": ADAM_LR_NET},
    {"params": phys_params,        "lr": ADAM_LR_PHYS},
])

# CosineAnnealingWarmRestarts: smooth LR decay with periodic warm restarts
# T_0 = restart period, T_mult = period multiplier after each restart
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    opt_adam, T_0=4000, T_mult=2, eta_min=1e-6
)

# ── Loss weights ─────────────────────────────────────────────────────────
W_PDE  = 10.0
W_BC   = 100.0
W_DATA = 1.0

print(f"Adam epochs: {ADAM_EPOCHS}")
print(f"LR net: {ADAM_LR_NET}, LR phys: {ADAM_LR_PHYS}")
print(f"Loss weights: PDE={W_PDE}, BC={W_BC}, Data={W_DATA}")

## 12. Checkpoint save/load utilities

In [ ]:
def save_adam_checkpoint(epoch, w_net, e_net, phys_params, optimizer, scheduler, history):
    path = os.path.join(CKPT_DIR, f"adam_epoch_{epoch}.pt")
    torch.save({
        "epoch": epoch,
        "w_net_state": w_net.state_dict(),
        "e_net_state": e_net.state_dict(),
        "phys_params": [p.data.clone() for p in phys_params],
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "history": history,
    }, path)
    print(f"  >> Adam checkpoint saved: epoch {epoch}")


def load_adam_checkpoint(w_net, e_net, phys_params, optimizer, scheduler):
    files = sorted(glob.glob(os.path.join(CKPT_DIR, "adam_epoch_*.pt")))
    if not files:
        return 0, {"loss": [], "pde": [], "bc": [], "data": []}
    path = files[-1]
    ckpt = torch.load(path, map_location=device)
    w_net.load_state_dict(ckpt["w_net_state"])
    e_net.load_state_dict(ckpt["e_net_state"])
    for p, saved in zip(phys_params, ckpt["phys_params"]):
        p.data.copy_(saved)
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    ep = ckpt["epoch"]
    hist = ckpt.get("history", {"loss": [], "pde": [], "bc": [], "data": []})
    print(f"  Resumed Adam from epoch {ep}")
    return ep, hist


def save_lbfgs_checkpoint(step, w_net, e_net, phys_params, optimizer):
    path = os.path.join(CKPT_DIR, f"lbfgs_step_{step}.pt")
    torch.save({
        "step": step,
        "w_net_state": w_net.state_dict(),
        "e_net_state": e_net.state_dict(),
        "phys_params": [p.data.clone() for p in phys_params],
        "optimizer_state": optimizer.state_dict(),
    }, path)
    print(f"  >> L-BFGS checkpoint saved: step {step}")


def load_lbfgs_checkpoint(w_net, e_net, phys_params, optimizer):
    files = sorted(glob.glob(os.path.join(CKPT_DIR, "lbfgs_step_*.pt")))
    if not files:
        return 0
    path = files[-1]
    ckpt = torch.load(path, map_location=device)
    w_net.load_state_dict(ckpt["w_net_state"])
    e_net.load_state_dict(ckpt["e_net_state"])
    for p, saved in zip(phys_params, ckpt["phys_params"]):
        p.data.copy_(saved)
    optimizer.load_state_dict(ckpt["optimizer_state"])
    step = ckpt["step"]
    print(f"  Resumed L-BFGS from step {step}")
    return step

## 13. Adam training phase (20,000 epochs)

In [ ]:
print("Starting Adam training...")

start_epoch, history = load_adam_checkpoint(w_net, e_net, phys_params, opt_adam, scheduler)

for epoch in range(start_epoch, ADAM_EPOCHS):
    opt_adam.zero_grad()
    loss, logs = pinn_loss(W_PDE, W_BC, W_DATA)
    loss.backward()
    opt_adam.step()
    scheduler.step()

    # ── Record history ───────────────────────────────────────────────────
    for key in ["loss", "pde", "bc", "data"]:
        history.setdefault(key, []).append(logs[key])

    # ── Print progress ───────────────────────────────────────────────────
    if epoch % 1000 == 0 or epoch == ADAM_EPOCHS - 1:
        eta_v, kx_v, kphi_v, F_v = get_params()
        lr_now = opt_adam.param_groups[0]["lr"]
        print(f"Epoch {epoch:5d} | Loss {logs['loss']:.3e} | PDE {logs['pde']:.3e} | "
              f"BC {logs['bc']:.3e} | Data {logs['data']:.3e} | "
              f"eta {eta_v.item():.5f} | kx {kx_v.item():.3e} | "
              f"kphi {kphi_v.item():.3e} | F {F_v.item():.4f} | lr {lr_now:.2e}")

    # ── Checkpoint to Drive ──────────────────────────────────────────────
    if (epoch > 0 and epoch % ADAM_CKPT_EVERY == 0) or epoch == ADAM_EPOCHS - 1:
        save_adam_checkpoint(epoch, w_net, e_net, phys_params, opt_adam, scheduler, history)

print("\nAdam phase complete.")

## 14. Adam training diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss history
ax = axes[0]
for key in ["loss", "pde", "bc", "data"]:
    ax.semilogy(history[key], label=key, alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log)")
ax.set_title("Loss history (Adam phase)")
ax.legend()
ax.grid(True, alpha=0.3)

# Parameter convergence
ax = axes[1]
eta_v, kx_v, kphi_v, F_v = get_params()
ax.set_title("Scalar parameter convergence")
ax.text(0.5, 0.5,
        f"eta  = {eta_v.item():.5f}\n"
        f"kx   = {kx_v.item():.3e}\n"
        f"kphi = {kphi_v.item():.3e}\n"
        f"F    = {F_v.item():.4f}",
        transform=ax.transAxes, fontsize=14, va='center', ha='center',
        fontfamily='monospace')
ax.axis('off')

plt.tight_layout()
plt.show()

## 15. L-BFGS refinement phase

Second-order optimization for fine-tuning. Each L-BFGS step involves
up to `max_iter` internal iterations with line search, so this is much
more expensive per step than Adam.

In [ ]:
print("Starting L-BFGS refinement...")

all_params = list(w_net.parameters()) + list(e_net.parameters()) + phys_params

opt_lbfgs = torch.optim.LBFGS(
    all_params, lr=0.1, max_iter=20,
    history_size=50, line_search_fn="strong_wolfe"
)

LBFGS_STEPS = 300

start_step = load_lbfgs_checkpoint(w_net, e_net, phys_params, opt_lbfgs)

for step in range(start_step, LBFGS_STEPS):
    def closure():
        opt_lbfgs.zero_grad()
        loss, _ = pinn_loss(W_PDE, W_BC, W_DATA)
        loss.backward()
        return loss

    opt_lbfgs.step(closure)

    # ── Print every 10 steps ─────────────────────────────────────────────
    if step % 10 == 0 or step == LBFGS_STEPS - 1:
        _, logs = pinn_loss(W_PDE, W_BC, W_DATA)
        eta_v, kx_v, kphi_v, F_v = get_params()
        print(f"L-BFGS {step:4d} | Loss {logs['loss']:.3e} | PDE {logs['pde']:.3e} | "
              f"Data {logs['data']:.3e} | eta {eta_v.item():.5f} | "
              f"kx {kx_v.item():.3e} | kphi {kphi_v.item():.3e} | F {F_v.item():.4f}")

    # ── Checkpoint to Drive ──────────────────────────────────────────────
    if step % LBFGS_CKPT_EVERY == 0 or step == LBFGS_STEPS - 1:
        save_lbfgs_checkpoint(step, w_net, e_net, phys_params, opt_lbfgs)

print("\nL-BFGS phase complete.")

## 16. Identified E(x) profile

In [ ]:
x_plot = torch.linspace(0, L, 200, dtype=DTYPE, device=device).unsqueeze(1)
with torch.no_grad():
    E_plot = e_net(normalize_x(x_plot)).cpu().numpy().flatten() / 1e9  # GPa

x_mm = x_plot.cpu().numpy().flatten() * 1e3

plt.figure(figsize=(12, 4))
plt.plot(x_mm, E_plot, 'b-', lw=2)
plt.axhline(E_INIT / 1e9, color='r', ls='--', alpha=0.5,
            label=f"Initial guess {E_INIT/1e9:.0f} GPa")
plt.axhline(E_MIN / 1e9, color='gray', ls=':', alpha=0.3,
            label=f"Bounds [{E_MIN/1e9:.0f}, {E_MAX/1e9:.0f}] GPa")
plt.axhline(E_MAX / 1e9, color='gray', ls=':', alpha=0.3)
plt.xlabel("Position along beam x [mm]")
plt.ylabel("E(x) [GPa]")
plt.title("Identified spatially varying Young's modulus E(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"E(x) range: {E_plot.min():.2f} - {E_plot.max():.2f} GPa")
print(f"E(x) mean:  {E_plot.mean():.2f} GPa")

## 17. PINN vs Experimental FRF

In [ ]:
# ── Dense frequency evaluation ────────────────────────────────────────────
f_eval = np.linspace(MIN_FREQ_HZ, MAX_FREQ_HZ, 1000)
w_eval = 2 * np.pi * f_eval
x_eval = np.full_like(f_eval, x_meas)

x_e = torch.tensor(x_eval, dtype=DTYPE, device=device).unsqueeze(1)
w_e = torch.tensor(w_eval, dtype=DTYPE, device=device).unsqueeze(1)

with torch.no_grad():
    Wr_e, Wi_e = w_net(normalize_x(x_e), normalize_omega(w_e))
    # V = i*omega*W -> V_r = -omega*Wi, V_i = omega*Wr
    Vr_pred = (-w_e * Wi_e).cpu().numpy().flatten()
    Vi_pred = (w_e * Wr_e).cpu().numpy().flatten()

V_pred = Vr_pred + 1j * Vi_pred

fig, axs = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Magnitude
axs[0].plot(f_train, 20 * np.log10(np.abs(V_train) + eps_db),
            'b-', lw=1, alpha=0.7, label="Experimental (smoothed)")
axs[0].plot(f_eval, 20 * np.log10(np.abs(V_pred) + eps_db),
            'r--', lw=1.5, label="PINN prediction")
axs[0].set_ylabel("Magnitude [dB]")
axs[0].legend()
axs[0].set_title("PINN vs Experimental FRF")
axs[0].grid(True, alpha=0.3)

# Phase
axs[1].plot(f_train, np.angle(V_train),
            'b-', lw=1, alpha=0.7, label="Experimental (smoothed)")
axs[1].plot(f_eval, np.angle(V_pred),
            'r--', lw=1.5, label="PINN prediction")
axs[1].set_ylabel("Phase [rad]")
axs[1].set_xlabel("Frequency [Hz]")
axs[1].legend()
axs[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Print final identified parameters ────────────────────────────────────
eta_v, kx_v, kphi_v, F_v = get_params()
print(f"\n{'='*60}")
print(f"IDENTIFIED PARAMETERS (v4)")
print(f"{'='*60}")
print(f"  eta   = {eta_v.item():.6f}")
print(f"  kx    = {kx_v.item():.4e}  N/m")
print(f"  kphi  = {kphi_v.item():.4e}  N*m/rad")
print(f"  F     = {F_v.item():.6f}  N")
print(f"  E(x)  = {E_plot.mean():.2f} GPa (mean)")
print(f"{'='*60}")

## 18. Save final trained model

In [ ]:
final_path = os.path.join(CKPT_DIR, "pinn_v4_final.pt")
torch.save({
    "w_net_state": w_net.state_dict(),
    "e_net_state": e_net.state_dict(),
    "phys_params": [p.data.clone() for p in phys_params],
    "beam_params": {"L": L, "b": b, "h": h, "rho": rho, "m_tip": m_tip},
    "preprocessing": {
        "MIN_FREQ_HZ": MIN_FREQ_HZ,
        "MAX_FREQ_HZ": MAX_FREQ_HZ,
        "N_train": N_train,
    },
    "E_bounds": {"E_MIN": E_MIN, "E_MAX": E_MAX},
}, final_path)
print(f"Final model saved to: {final_path}")